In [83]:
%load_ext autoreload
%autoreload 2

import itertools
from math import prod

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from assignment_1.modules.models import fit_bpr, bpr_scorer
from assignment_1.modules.util import calc_pair_accuracy, load_csv, random_split

DATA_DIR = "../data"  # notebook runs from notebooks/

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [84]:
train = load_csv("tasks_1_2_train.csv", DATA_DIR)
test = load_csv("task_2_test.csv", DATA_DIR)


In [85]:
train.head(5)

,user,item,timestamp,rating
0,u6615992338,i4026940741,1387439652,4
1,u1474879069,i5361500840,1358838765,5
2,u1060791114,i1810200320,1429854010,5
3,u1309139104,i4536388280,1409468967,4
4,u3188006761,i2217243877,1439792233,3


In [86]:
test.head(5)

,user,item1,item2,which
0,u6589235933,i6679693485,i3513531206,NaN
1,u5036346542,i9742011999,i4000434505,NaN
2,u2543223670,i8553129811,i8159127157,NaN
3,u6680502243,i5391486098,i3623313194,NaN
4,u2041216719,i5735921315,i5729684422,NaN


In [87]:
# interactions per item in training
pop = train["item"].value_counts()           
p1 = test["item1"].map(pop).fillna(0)
p2 = test["item2"].map(pop).fillna(0)

print(f"unseen users: {(~test.user.isin(train.user)).mean():.1%}")
print(f"unseen item1: {(p1 == 0).mean():.1%} | unseen item2: {(p2 == 0).mean():.1%}")
print(f"median popularity: item1 = {p1.median():.0f}, item2 = {p2.median():.0f}")
print(f"P(item1 more popular) = {(p1 > p2).mean():.3f} | ties = {(p1 == p2).mean():.3f}")

print("\nreference medians:")
print(f"  item drawn uniformly from catalog: {pop.median():.0f}")
print(f"  item drawn ∝ interactions:         {train['item'].map(pop).median():.0f}")

unseen users: 0.0%
unseen item1: 0.0% | unseen item2: 0.0%
median popularity: item1 = 58, item2 = 57
P(item1 more popular) = 0.497 | ties = 0.007

reference medians:
  item drawn uniformly from catalog: 18
  item drawn ∝ interactions:         59


In [88]:
tr, val = random_split(train, val_frac=0.10, seed=0)
seen = val.user.isin(tr.user) & val.item.isin(tr.item)
val = val[seen].reset_index(drop=True)

print(f"kept {seen.mean():.1%} of validation rows")

kept 100.0% of validation rows


In [89]:
history = train.groupby("user")["item"].apply(set).to_dict() # dict[user] = set[item]
users = val.user.to_numpy() # pair k -> user ID
train_items = train.item.to_numpy() # train row -> item ID


# first draw: one random train row per validation pair, take its book
rng = np.random.default_rng(0)
idx = rng.integers(len(train), size=len(val))
decoys = train.item.to_numpy()[idx]

assert len(decoys) == len(val)

In [90]:
print(users.shape)
print(train_items.shape)


(186164,)
(1861643,)


In [91]:
bad = np.array([i for i, d in enumerate(decoys) if d in history[users[i]]])

print(len(bad), len(bad) / len(val))

while bad.size > 0:
    # pull a new random item for each bad pair
    for i in bad:
        decoys[i] = train_items[rng.integers(len(train))]

    # keep only the ones that collided again
    bad = np.array([i for i in bad if decoys[i] in history[users[i]]], dtype=int)
    print(len(bad))



3723 0.019998495949807696
181
14
2
1
0


In [92]:
assert not any(d in history[u] for u, d in zip(users, decoys))
assert (val.item.to_numpy() != decoys).all()

In [93]:
real = val.item.to_numpy()
real_first = rng.random(len(val)) < 0.5      # coin flip per pair

pairs = pd.DataFrame(index=range(len(val)), columns=["user", "item1", "item2", "which"])
pairs["user"]  = users
pairs["which"] = real
pairs["item1"] = np.where(real_first, val.item, decoys)
pairs["item2"] = np.where(real_first, decoys, val.item)

In [94]:
assert (pairs.item1 == pairs.item2).sum() == 0         # should be 0
frac = (pairs.which == pairs.item1).mean()
assert abs(frac - 0.5) < 0.01, f"real item in item1 {frac:.3f} of the time"      # should be ≈ 0.5
pairs.head()

,user,item1,item2,which
0,u1603088681,i9960281822,i3791430712,i9960281822
1,u3898574929,i9464761793,i0695900379,i9464761793
2,u7753644730,i8642457945,i5114999221,i5114999221
3,u2943949681,i4451239709,i2201875856,i4451239709
4,u8122465982,i1240061652,i9674039066,i1240061652


In [95]:
pop = train.item.value_counts()

calc_pair_accuracy(pairs, lambda u, i: rng.random(len(i)))                       # random     ≈ 0.50

0.499033110590662

In [96]:
calc_pair_accuracy(pairs, lambda u, i: np.zeros(len(i)))                         # constant   ≈ 0.50 (all ties -> item1)

0.5018478330934015

In [97]:
calc_pair_accuracy(pairs, lambda u, i: pd.Series(i).map(pop).fillna(0).to_numpy()) 

0.5056401882211383

In [98]:
fit = fit_bpr(tr, factors=5, regularization=0.01, learning_rate=0.01, iterations=100)
base = calc_pair_accuracy(pairs, bpr_scorer(fit))   # ≈ 0.877

In [ ]:
grid1 = {
    "factors":        [5, 20, 50],
    "regularization": [0.001, 0.01, 0.1],
    "learning_rate":  [0.01],
    "iterations":     [100],
}

grid2 = {
    "factors":        [50, 100, 200],
    "regularization": [0.003, 0.01, 0.03],
    "learning_rate":  [0.01],
    "iterations":     [100],
}

rows = []
for values in tqdm(itertools.product(*grid2.values()), total=prod(map(len, grid2.values()))):
    params = dict(zip(grid2, values))          # {"factors": 20, "regularization": 0.01, ...}
    fit = fit_bpr(tr, **params)
    acc = calc_pair_accuracy(pairs, bpr_scorer(fit))
    rows.append({**params, "val_acc": acc, "vs_base_%": (acc / base - 1) * 100})

results = pd.DataFrame(rows).sort_values("val_acc", ascending=False)
results

  0%|          | 0/9 [00:00<?, ?it/s]